In [1]:
!nvidia-smi

!python --version


Wed Dec 10 00:38:55 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Bạn có thể đổi tên folder nếu muốn
!mkdir -p "/content/drive/MyDrive/stylegan3_project"
!mkdir -p "/content/drive/MyDrive/stylegan3_project/out_test"
!mkdir -p "/content/drive/MyDrive/stylegan3_project/gen_raw"
!mkdir -p "/content/drive/MyDrive/stylegan3_project/filtered/asian"
!mkdir -p "/content/drive/MyDrive/stylegan3_project/filtered/child"


In [3]:
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
!bash miniconda.sh -b -p /content/miniconda
!/content/miniconda/bin/conda --version


PREFIX=/content/miniconda
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /content/miniconda
conda 25.9.1


In [4]:
!/content/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!/content/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r


accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r


In [5]:
!/content/miniconda/bin/conda create -y -n sg3 python=3.8
!/content/miniconda/bin/conda env list


Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: - \ | / - \ | / - \ | / - done
Channels:
 - defaults
Platform: linux-64
Solving environment: | done


==> WARNING: A newer version of conda exists. <==
    current version: 25.9.1
    latest version: 25.11.0

Please update conda by running

    $ conda update -n base -c defaults conda



## Package Plan ##

  environment location: /content/miniconda/envs/sg3

  added / updated specs:
    - python=3.8


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2025.12.2  |       h06a4308_0         125 KB
    pip-24.2                   |   py38h06a4308_0         2.2 MB
    python-3.8.20              |       he870216_0        23.8 MB
    setuptools-75.1.0          |   py38h06a4308_0         1.7 MB
    wheel-0.44.0               |   py38h06a4308_0         108 KB
    -----------------

In [6]:
# update pip
!/content/miniconda/bin/conda run -n sg3 python -m pip install -U pip

# deps cơ bản + các gói hay thiếu khi load network
!/content/miniconda/bin/conda run -n sg3 pip install ninja pyspng imageio-ffmpeg click scipy pillow numpy

# pytorch CUDA cho T4 (cu117)
!/content/miniconda/bin/conda run -n sg3 pip install \
  torch==1.13.1+cu117 torchvision==0.14.1+cu117 \
  --index-url https://download.pytorch.org/whl/cu117

# kiểm tra GPU trong env
!/content/miniconda/bin/conda run -n sg3 python -c "import torch; print('torch',torch.__version__); print('cuda?',torch.cuda.is_available()); print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 88.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.2
    Uninstalling pip-24.2:
      Successfully uninstalled pip-24.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.9/26.9 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 156.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 140.7 MB/s eta 0:00:00

Looking in indexes: https://download.pytorch.org/whl/cu117
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 11.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.3/24.3 MB 117.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of typing-extensions to determine which version is compatible with other requirements. This could take a while.

torch 1.13.1+cu117
cuda? True
gpu Tesla T4



In [7]:
!rm -rf stylegan3
!git clone https://github.com/NVlabs/stylegan3.git

# cài requirements của repo
!/content/miniconda/bin/conda run -n sg3 pip install -r stylegan3/requirements.txt

# xoá cache extension để build sạch (tránh lỗi bias_act_plugin)
!rm -rf ~/.cache/torch_extensions


Cloning into 'stylegan3'...
remote: Enumerating objects: 212, done.
remote: Counting objects: 100% (163/163), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 212 (delta 99), reused 90 (delta 90), pack-reused 49 (from 1)
Receiving objects: 100% (212/212), 4.16 MiB | 16.84 MiB/s, done.
Resolving deltas: 100% (108/108), done.
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'stylegan3/requirements.txt'

ERROR conda.cli.main_run:execute(127): `conda run pip install -r stylegan3/requirements.txt` failed. (See above for error)


In [ ]:
!/content/miniconda/bin/conda run -n sg3 bash -lc 'cd stylegan3 && \
python gen_images.py \
  --outdir="/content/out_test" \
  --trunc=0.3 --seeds=6601-6601 \
  --network="https://api.ngc.nvidia.com/v2/models/nvidia/research/stylegan3/versions/1/files/stylegan3-r-ffhq-1024x1024.pkl"'


In [ ]:
!/content/miniconda/bin/conda run -n sg3 bash -lc 'cd stylegan3 && \
python gen_images.py \
  --outdir="/content/5000_01_test" \
  --trunc=0.4 --seeds=501-5500 \
  --network="https://api.ngc.nvidia.com/v2/models/nvidia/research/stylegan3/versions/1/files/stylegan3-r-ffhq-1024x1024.pkl"'


In [9]:
!zip -r /content/500.zip /content/100_test/

  adding: content/100_test/ (stored 0%)
  adding: content/100_test/seed0153.png (deflated 0%)
  adding: content/100_test/seed0213.png (deflated 0%)
  adding: content/100_test/seed0027.png (deflated 0%)
  adding: content/100_test/seed0425.png (deflated 0%)
  adding: content/100_test/seed0016.png (deflated 0%)
  adding: content/100_test/seed0222.png (deflated 0%)
  adding: content/100_test/seed0395.png (deflated 0%)
  adding: content/100_test/seed0381.png (deflated 0%)
  adding: content/100_test/seed0198.png (deflated 0%)
  adding: content/100_test/seed0279.png (deflated 0%)
  adding: content/100_test/seed0334.png (deflated 0%)
  adding: content/100_test/seed0477.png (deflated 0%)
  adding: content/100_test/seed0384.png (deflated 0%)
  adding: content/100_test/seed0207.png (deflated 0%)
  adding: content/100_test/seed0113.png (deflated 0%)
  adding: content/100_test/seed0344.png (deflated 0%)
  adding: content/100_test/seed0390.png (deflated 0%)
  adding: content/100_test/seed0038.png (d

In [11]:
from google.colab import files
files.download('/content/500.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#LỌC

In [12]:
!/content/miniconda/bin/conda run -n sg3 pip install -U insightface onnxruntime-gpu opencv-python transformers


  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached cython-3.2.2-cp38-cp38-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (4.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.2/226.2 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.0/806.0 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 785.1/785.1 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 10.4 MB

In [ ]:
#Lọc và copy sang ổ Notebook

In [ ]:
!pip install insightface==0.7.3 onnxruntime-gpu

In [ ]:
!/content/miniconda/bin/conda run -n sg3 python - << "PY"
import os, re, csv, shutil, glob
import cv2

# ====== CONFIG ======
# Nếu bạn gen 10k test:
IN_GLOB = "/content/drive/MyDrive/stylegan3_project/gen_raw/*.png"

# Nếu bạn gen 100k chunk:
#IN_GLOB = "/content/drive/MyDrive/stylegan3_project/gen_100k/**/*.png"

FILTER_ASIAN = True
FILTER_CHILD = True
CHILD_MAX_AGE = 12  # đổi 18 nếu bạn muốn <18

# OUTPUT: copy về ổ Colab
OUT_ASIAN = "/content/filtered/asian"
OUT_CHILD = "/content/filtered/child"
CSV_PATH  = "/content/filtered/selected.csv"

os.makedirs(OUT_ASIAN, exist_ok=True)
os.makedirs(OUT_CHILD, exist_ok=True)
os.makedirs(os.path.dirname(CSV_PATH), exist_ok=True)

# ====== seed parser ======
seed_re = re.compile(r"(\d+)")
def parse_seed(path):
    base = os.path.basename(path)
    m = seed_re.findall(base)
    return int(m[0]) if m else None

# ====== InsightFace (age) ======
from insightface.app import FaceAnalysis
app = FaceAnalysis(allowed_modules=["detection","genderage"])
app.prepare(ctx_id=0, det_size=(640,640))

# ====== CLIP race classifier (practical) ======
from transformers import CLIPProcessor, CLIPModel
import torch
import PIL.Image

model_id = "syntheticbot/clip-face-attribute-classifier"
clip_model = CLIPModel.from_pretrained(model_id)
clip_proc  = CLIPProcessor.from_pretrained(model_id)
clip_model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model.to(device)

RACE_LABELS = ["White","Black","Indian","East Asian","Southeast Asian","Middle Eastern","Latino"]
ASIAN_SET = {"East Asian","Southeast Asian"}

def predict_race_bgr(img_bgr):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil = PIL.Image.fromarray(img_rgb)
    inputs = clip_proc(text=RACE_LABELS, images=pil, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        logits = clip_model(**inputs).logits_per_image[0]
        probs = logits.softmax(dim=-1).detach().cpu().numpy()
    idx = int(probs.argmax())
    return RACE_LABELS[idx], float(probs[idx])

# ====== Main loop ======
rows = []
kept_asian = kept_child = scanned = 0

for path in sorted(glob.glob(IN_GLOB, recursive=True)):
    scanned += 1
    img = cv2.imread(path)
    if img is None:
        continue

    faces = app.get(img)
    if not faces:
        continue

    # face lớn nhất
    face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
    age = float(getattr(face, "age", -1))

    race, race_conf = predict_race_bgr(img)

    is_child = (age >= 0 and age <= CHILD_MAX_AGE) if FILTER_CHILD else False
    is_asian = (race in ASIAN_SET) if FILTER_ASIAN else False

    # Copy về ổ Colab
    if is_child:
        kept_child += 1
        shutil.copy2(path, os.path.join(OUT_CHILD, os.path.basename(path)))
    if is_asian:
        kept_asian += 1
        shutil.copy2(path, os.path.join(OUT_ASIAN, os.path.basename(path)))

    if is_child or is_asian:
        rows.append([os.path.basename(path), parse_seed(path), age, race, race_conf, int(is_asian), int(is_child)])

print("Scanned:", scanned)
print("Kept asian:", kept_asian, "->", OUT_ASIAN)
print("Kept child:", kept_child, "->", OUT_CHILD)

with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["file","seed","age_est","race_pred","race_conf","is_asian","is_child"])
    w.writerows(rows)

print("Wrote:", CSV_PATH)
PY


In [ ]:
#Lọc 2
!cp -r /content/gen_raw/ /content/drive/MyDrive/

In [ ]:
!/content/miniconda/bin/conda run -n sg3 python - << "PY"
import os, re, csv, shutil, glob
import cv2

# ================== INPUT (Drive) ==================
# Nếu bạn gen 100k theo chunk:
IN_GLOB = "/content/gen_raw/**/*.png"
# Nếu bạn gen 1 folder:
# IN_GLOB = "/content/drive/MyDrive/stylegan3_project/gen_raw/*.png"

# ================== THRESHOLDS ==================
CHILD_MAX_AGE   = 8
ADULT_MIN_AGE   = 18
ELDERLY_MIN_AGE = 65

# ================== OUTPUT (Colab disk) ==================
OUTROOT  = "/content/filtered_asian"
OUT_CHILD = os.path.join(OUTROOT, "CHILD8")
OUT_ELD   = os.path.join(OUTROOT, "Elderly")
OUT_MALE  = os.path.join(OUTROOT, "Male")
OUT_FEMA  = os.path.join(OUTROOT, "Female")
CSV_PATH  = os.path.join(OUTROOT, "selected_asian.csv")

for d in [OUT_CHILD, OUT_ELD, OUT_MALE, OUT_FEMA]:
    os.makedirs(d, exist_ok=True)
os.makedirs(os.path.dirname(CSV_PATH), exist_ok=True)

# ================== seed parser ==================
seed_re = re.compile(r"(\d+)")
def parse_seed(path):
    base = os.path.basename(path)
    m = seed_re.findall(base)
    return int(m[0]) if m else None

# ================== InsightFace (age+gender) ==================
from insightface.app import FaceAnalysis
app = FaceAnalysis(allowed_modules=["detection","genderage"])
app.prepare(ctx_id=0, det_size=(640,640))

def infer_age_gender(img_bgr):
    faces = app.get(img_bgr)
    if not faces:
        return None
    # chọn face lớn nhất
    face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
    age = float(getattr(face, "age", -1))
    gender_id = int(getattr(face, "gender", -1))  # thường: 0=female, 1=male
    gender = "F" if gender_id == 0 else ("M" if gender_id == 1 else "U")
    return age, gender

# ================== CLIP race classifier (Asian gate) ==================
from transformers import CLIPProcessor, CLIPModel
import torch
import PIL.Image

model_id = "syntheticbot/clip-face-attribute-classifier"
clip_model = CLIPModel.from_pretrained(model_id)
clip_proc  = CLIPProcessor.from_pretrained(model_id)
clip_model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model.to(device)

RACE_LABELS = ["White","Black","Indian","East Asian","Southeast Asian","Middle Eastern","Latino"]
ASIAN_SET = {"East Asian","Southeast Asian"}

def predict_race_bgr(img_bgr):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil = PIL.Image.fromarray(img_rgb)
    inputs = clip_proc(text=RACE_LABELS, images=pil, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        logits = clip_model(**inputs).logits_per_image[0]
        probs = logits.softmax(dim=-1).detach().cpu().numpy()
    idx = int(probs.argmax())
    return RACE_LABELS[idx], float(probs[idx])

# ================== MAIN ==================
scanned = 0
asian_pass = 0
kept = {"CHILD8":0, "Elderly":0, "MaleAdult":0, "FemaleAdult":0}
rows = []

for path in sorted(glob.glob(IN_GLOB, recursive=True)):
    scanned += 1
    img = cv2.imread(path)
    if img is None:
        continue

    # Gate: Asian?
    race, race_conf = predict_race_bgr(img)
    if race not in ASIAN_SET:
        continue
    asian_pass += 1

    ag = infer_age_gender(img)
    if ag is None:
        continue
    age, gender = ag

    base = os.path.basename(path)
    seed = parse_seed(path)

    # Bucket rules (ưu tiên CHILD8/Elderly trước; adult mới chia gender)
    bucket = None
    if age >= 0 and age <= CHILD_MAX_AGE:
        bucket = "CHILD8"
        dst = os.path.join(OUT_CHILD, base)
        kept["CHILD8"] += 1
        shutil.copy2(path, dst)
    elif age >= ELDERLY_MIN_AGE:
        bucket = "Elderly"
        dst = os.path.join(OUT_ELD, base)
        kept["Elderly"] += 1
        shutil.copy2(path, dst)
    elif age >= ADULT_MIN_AGE and age < ELDERLY_MIN_AGE:
        # Adult: chia gender
        if gender == "M":
            bucket = "Male"
            dst = os.path.join(OUT_MALE, base)
            kept["MaleAdult"] += 1
            shutil.copy2(path, dst)
        elif gender == "F":
            bucket = "Female"
            dst = os.path.join(OUT_FEMA, base)
            kept["FemaleAdult"] += 1
            shutil.copy2(path, dst)
        else:
            # gender unknown -> bỏ qua hoặc bạn có thể tạo folder "AdultUnknown"
            bucket = "AdultUnknown"
            dst = None
    else:
        bucket = "Other"  # ví dụ 9-17, hoặc age=-1
        dst = None

    # Ghi CSV cho mọi ảnh Asian pass (kể cả không bucket) để bạn audit
    rows.append([base, seed, age, gender, race, race_conf, bucket, dst])

print("Scanned:", scanned)
print("Asian passed:", asian_pass)
print("Kept:", kept)
print("Output root:", OUTROOT)

with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["file","seed","age_est","gender","race_pred","race_conf","bucket","dst_path"])
    w.writerows(rows)

print("Wrote:", CSV_PATH)
PY


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Lọc - Có phân thư mục con, giới tính, trẻ em

In [16]:
!pip install insightface==0.7.3 onnxruntime

  Using cached insightface-0.7.3.tar.gz (439 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached coloredlogs-15.0.1-py2.py3-none-any.whl.metadata (12 kB)
  Using cached humanfriendly-10.0-py2.py3-none-any.whl.metadata (9.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 119.5 MB/s eta 0:00:00
Using cached coloredlogs-15.0.1-py2.py3-none-any.whl (46 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 32.8 MB/s eta 0:00:00
Using cached humanfriendly-10.0-py2.py3-none-any.whl (86 kB)
  Created wheel for insightface: filename=insightface-0.7.3-cp312-cp312-linux_x86_64.whl size=1071353 sha256=891a0b13f9006037572765ab1f3f80aca41a6e1952b0350f2c43c8d127d0742c
  Stored in directory: /root/.cache/pip/wheels/73/3c/e2/6d4815e8a8b33a2006554d65ce0d1f973e768f4c7a222fa675
Successfully built insightface


In [18]:
import os, re, csv, shutil, glob
import cv2
import numpy as np

# ================== INPUT (Drive) ==================
IN_GLOB = "/content/100_test/**/*.png"
# IN_GLOB = "/content/drive/MyDrive/stylegan3_project/gen_raw/*.png"

# ================== THRESHOLDS ==================
CHILD_MAX_AGE   = 8
ADULT_MIN_AGE   = 18
ELDERLY_MIN_AGE = 65

# CLIP gate: nếu điểm Asian thấp quá thì bỏ
ASIAN_CONF_MIN = 0.35  # bạn có thể thử 0.30 -> 0.45 tuỳ dữ liệu

# ================== OUTPUT (Colab disk) ==================
OUTROOT   = "/content/filtered_asian"
OUT_CHILD = os.path.join(OUTROOT, "CHILD8")
OUT_ELD   = os.path.join(OUTROOT, "Elderly")
OUT_MALE  = os.path.join(OUTROOT, "Male")
OUT_FEMA  = os.path.join(OUTROOT, "Female")
CSV_PATH  = os.path.join(OUTROOT, "selected_asian.csv")

for d in [OUT_CHILD, OUT_ELD, OUT_MALE, OUT_FEMA]:
    os.makedirs(d, exist_ok=True)

# ================== seed parser ==================
seed_re = re.compile(r"(\d+)")
def parse_seed(path):
    base = os.path.basename(path)
    m = seed_re.findall(base)
    return int(m[0]) if m else None

# ================== InsightFace (age+gender) ==================
from insightface.app import FaceAnalysis
app = FaceAnalysis(allowed_modules=["detection","genderage"])
app.prepare(ctx_id=0, det_size=(640,640))

def infer_age_gender(img_bgr):
    faces = app.get(img_bgr)
    if not faces:
        return None
    face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
    age = float(getattr(face, "age", -1))
    gender_id = int(getattr(face, "gender", -1))  # 0=female, 1=male
    gender = "F" if gender_id == 0 else ("M" if gender_id == 1 else "U")
    return age, gender

# ================== CLIP zero-shot Asian gate ==================
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

PROMPTS = [
    "a photo of an East Asian person",
    "a photo of a Southeast Asian person",
    "a photo of a person",
    "a photo of a White person",
    "a photo of a Black person",
    "a photo of an Indian person",
]

ASIAN_IDXS = [0, 1]  # East + Southeast

def clip_asian_score(img_bgr):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    inputs = proc(text=PROMPTS, images=pil, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        logits = clip(**inputs).logits_per_image[0]
        probs = logits.softmax(dim=-1).detach().cpu().numpy()
    asian_score = float(probs[0] + probs[1])
    best_idx = int(np.argmax(probs))
    best_prompt = PROMPTS[best_idx]
    best_prob = float(probs[best_idx])
    return asian_score, best_prompt, best_prob

# ================== MAIN ==================
scanned = 0
asian_pass = 0
kept = {"CHILD8":0, "Elderly":0, "MaleAdult":0, "FemaleAdult":0}
rows = []

for path in sorted(glob.glob(IN_GLOB, recursive=True)):
    scanned += 1
    img = cv2.imread(path)
    if img is None:
        continue

    asian_score, best_prompt, best_prob = clip_asian_score(img)
    if asian_score < ASIAN_CONF_MIN:
        continue
    asian_pass += 1

    ag = infer_age_gender(img)
    if ag is None:
        continue
    age, gender = ag

    base = os.path.basename(path)
    seed = parse_seed(path)

    bucket = None
    dst = None

    if age >= 0 and age <= CHILD_MAX_AGE:
        bucket = "CHILD8"
        dst = os.path.join(OUT_CHILD, base)
        kept["CHILD8"] += 1
    elif age >= ELDERLY_MIN_AGE:
        bucket = "Elderly"
        dst = os.path.join(OUT_ELD, base)
        kept["Elderly"] += 1
    elif age >= ADULT_MIN_AGE and age < ELDERLY_MIN_AGE:
        if gender == "M":
            bucket = "Male"
            dst = os.path.join(OUT_MALE, base)
            kept["MaleAdult"] += 1
        elif gender == "F":
            bucket = "Female"
            dst = os.path.join(OUT_FEMA, base)
            kept["FemaleAdult"] += 1
        else:
            bucket = "AdultUnknown"
    else:
        bucket = "Other"  # 9-17 hoặc age=-1

    if dst is not None:
        shutil.copy2(path, dst)

    rows.append([base, seed, age, gender, asian_score, ASIAN_CONF_MIN, best_prompt, best_prob, bucket, dst])

print("Scanned:", scanned)
print("Asian passed (asian_score>=min):", asian_pass, "min=", ASIAN_CONF_MIN)
print("Kept:", kept)
print("Output root:", OUTROOT)

with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["file","seed","age_est","gender","asian_score","asian_conf_min","best_prompt","best_prob","bucket","dst_path"])
    w.writerows(rows)

print("Wrote:", CSV_PATH)

download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:05<00:00, 48741.73KB/s]
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition
set det-size: (640, 640)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Scanned: 501
Asian passed (asian_score>=min): 44 min= 0.35
Kept: {'CHILD8': 1, 'Elderly': 0, 'MaleAdult': 2, 'FemaleAdult': 11}
Output root: /content/filtered_asian
Wrote: /content/filtered_asian/selected_asian.csv


In [32]:
!rm -rf /content/filtered_asian


In [21]:
import os, re, csv, shutil, glob
import cv2
import numpy as np

# ================== INPUT ==================
IN_GLOB = "/content/100_test/**/*.png"
# IN_GLOB = "/content/100_test/*.png"

# ================== THRESHOLDS ==================
ASIAN_CONF_MIN = 0.35  # thử 0.25-0.45 tuỳ dữ liệu

# Age bins (theo yêu cầu đã chốt)
# - CHILD8: <=8
# - 9-16: 9..16
# - 17-25: 17..25
# - 26-35, 36-45, 46-55, 56-65
# - Elderly: >=66
AGE_BINS = [
    ("CHILD8",   0,   8),
    ("9-16",     9,  16),
    ("17-25",   17,  25),
    ("26-35",   26,  35),
    ("36-45",   36,  45),
    ("46-55",   46,  55),
    ("56-65",   56,  65),
    ("Elderly", 66, 200),
]

# ================== OUTPUT (Colab disk) ==================
OUTROOT  = "/content/filtered_asian"
CSV_PATH = os.path.join(OUTROOT, "selected_asian.csv")

# Tạo toàn bộ folder: <OUTROOT>/<BIN>/<Male|Female>
for bin_name, _, _ in AGE_BINS:
    os.makedirs(os.path.join(OUTROOT, bin_name, "Male"), exist_ok=True)
    os.makedirs(os.path.join(OUTROOT, bin_name, "Female"), exist_ok=True)

# Nếu gender unknown
os.makedirs(os.path.join(OUTROOT, "UnknownGender"), exist_ok=True)

# ================== seed parser ==================
seed_re = re.compile(r"(\d+)")
def parse_seed(path):
    base = os.path.basename(path)
    m = seed_re.findall(base)
    return int(m[0]) if m else None

# ================== InsightFace (age+gender) ==================
from insightface.app import FaceAnalysis
app = FaceAnalysis(allowed_modules=["detection","genderage"])
app.prepare(ctx_id=0, det_size=(640,640))

def infer_age_gender(img_bgr):
    faces = app.get(img_bgr)
    if not faces:
        return None
    face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
    age = float(getattr(face, "age", -1))
    gender_id = int(getattr(face, "gender", -1))  # thường: 0=female, 1=male
    gender = "F" if gender_id == 0 else ("M" if gender_id == 1 else "U")
    return age, gender

# ================== CLIP zero-shot Asian gate ==================
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

PROMPTS = [
    "a photo of an East Asian person",
    "a photo of a Southeast Asian person",
    "a photo of a person",
    "a photo of a White person",
    "a photo of a Black person",
    "a photo of an Indian person",
]

def clip_asian_score(img_bgr):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    inputs = proc(text=PROMPTS, images=pil, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        logits = clip(**inputs).logits_per_image[0]
        probs = logits.softmax(dim=-1).detach().cpu().numpy()
    asian_score = float(probs[0] + probs[1])  # East + Southeast
    best_idx = int(np.argmax(probs))
    best_prompt = PROMPTS[best_idx]
    best_prob = float(probs[best_idx])
    return asian_score, best_prompt, best_prob

# ================== Age bin helper ==================
def get_age_bin(age: float):
    if age is None or age < 0:
        return None
    if age <= 8:
        return "CHILD8"
    for name, lo, hi in AGE_BINS:
        if name == "CHILD8":
            continue
        if lo <= age <= hi:
            return name
    return None

# ================== MAIN ==================
scanned = 0
asian_pass = 0

kept_counts = {f"{b}/{g}": 0 for (b,_,_) in AGE_BINS for g in ["Male", "Female"]}
unknown_gender = 0
unknown_age = 0
no_face = 0

rows = []

for path in sorted(glob.glob(IN_GLOB, recursive=True)):
    scanned += 1
    img = cv2.imread(path)
    if img is None:
        continue

    asian_score, best_prompt, best_prob = clip_asian_score(img)
    if asian_score < ASIAN_CONF_MIN:
        continue
    asian_pass += 1

    ag = infer_age_gender(img)
    if ag is None:
        no_face += 1
        continue
    age, gender = ag

    base = os.path.basename(path)
    seed = parse_seed(path)

    bin_name = get_age_bin(age)
    dst = None
    bucket = None

    if bin_name is None:
        unknown_age += 1
        bucket = "UnknownAge"
    else:
        bucket = bin_name

        if gender == "M":
            gender_folder = "Male"
        elif gender == "F":
            gender_folder = "Female"
        else:
            gender_folder = None

        if gender_folder is None:
            unknown_gender += 1
            dst = os.path.join(OUTROOT, "UnknownGender", base)
            shutil.copy2(path, dst)
        else:
            dst = os.path.join(OUTROOT, bucket, gender_folder, base)
            shutil.copy2(path, dst)
            kept_counts[f"{bucket}/{gender_folder}"] += 1

    rows.append([base, seed, age, gender, asian_score, ASIAN_CONF_MIN, best_prompt, best_prob, bucket, dst])

print("Scanned:", scanned)
print("Asian passed:", asian_pass, "min=", ASIAN_CONF_MIN)
print("No face detected:", no_face)
print("Unknown age:", unknown_age, "| Unknown gender:", unknown_gender)
print("Output root:", OUTROOT)

for k, v in kept_counts.items():
    if v > 0:
        print(f"{k}: {v}")

with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow([
        "file", "seed", "age_est", "gender",
        "asian_score", "asian_conf_min",
        "best_prompt", "best_prob",
        "bucket", "dst_path"
    ])
    w.writerows(rows)

print("Wrote:", CSV_PATH)


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition
set det-size: (640, 640)
Scanned: 501
Asian passed: 44 min= 0.35
No face detected: 29
Unknown age: 0 | Unknown gender: 0
Outp

In [ ]:
#Có lọc Asian

In [31]:
import os, re, csv, shutil, glob
import cv2
import numpy as np

# ================== INPUT ==================
IN_GLOB = "/content/100_test/**/*.png"

# ================== THRESHOLDS ==================
ASIAN_CONF_MIN   = 0.05   # Asian gate (CLIP)

# CHILD8 tight
CHILDLIKE_MIN    = 0.60   # tăng để bắt chặt hơn
ADULTLIKE_MAX_IN_CHILD = 0.30  # nếu CLIP thấy adult cao quá thì loại khỏi CHILD8

# Adult bins anti-child filter (để chặn 9-18 lọt vào 19-45)
ADULT_CHILD_MAX  = 0.25   # child_score phải <= ngưỡng này mới được vào bin adult
ADULTLIKE_MIN    = 0.35   # adult_score phải >= ngưỡng này mới được vào bin adult (tuỳ chọn nhưng rất hữu ích)

# Elderly (tuỳ chọn)
ELDERLYLIKE_MIN  = 0.25   # elderly_score tối thiểu để vào Elderly (không quá gắt)

# ================== AGE BINS ==================
AGE_BINS = [
    ("CHILD8",   0,   8),
    ("9-18",     9,  18),
    ("19-45",   19,  45),
    ("46-65",   46,  65),
    ("Elderly", 66, 200),
]
ADULT_BINS = {"19-45","46-65","Elderly"}

# ================== OUTPUT ==================
OUTROOT  = "/content/filtered_asian"
CSV_PATH = os.path.join(OUTROOT, "selected_asian.csv")

for bin_name, _, _ in AGE_BINS:
    os.makedirs(os.path.join(OUTROOT, bin_name, "Male"), exist_ok=True)
    os.makedirs(os.path.join(OUTROOT, bin_name, "Female"), exist_ok=True)

# ================== seed parser ==================
seed_re = re.compile(r"(\d+)")
def parse_seed(path):
    base = os.path.basename(path)
    m = seed_re.findall(base)
    return int(m[0]) if m else None

# ================== InsightFace (age+gender) ==================
from insightface.app import FaceAnalysis
app = FaceAnalysis(allowed_modules=["detection","genderage"])
app.prepare(ctx_id=0, det_size=(640,640))

def infer_age_gender(img_bgr):
    faces = app.get(img_bgr)
    if not faces:
        return None
    face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
    age = float(getattr(face, "age", -1))
    gender_id = int(getattr(face, "gender", -1))
    gender = "F" if gender_id == 0 else ("M" if gender_id == 1 else "U")
    return age, gender

# ================== CLIP ==================
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

ASIAN_PROMPTS = [
    "a photo of an East Asian person",
    "a photo of a Southeast Asian person",
    "a photo of a person",
    "a photo of a White person",
    "a photo of a Black person",
    "a photo of an Indian person",
]

AGESTYLE_PROMPTS = [
    "a photo of a child",
    "a photo of a young kid",
    "a photo of a teenage girl",
    "a photo of a teenage boy",
    "a photo of an adult person",
    "a photo of an elderly person",
]

def _clip_probs(img_bgr, prompts):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    inputs = proc(text=prompts, images=pil, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        logits = clip(**inputs).logits_per_image[0]
        probs = logits.softmax(dim=-1).detach().cpu().numpy()
    return probs

def clip_asian_score(img_bgr):
    probs = _clip_probs(img_bgr, ASIAN_PROMPTS)
    asian_score = float(probs[0] + probs[1])
    best_idx = int(np.argmax(probs))
    best_prompt = ASIAN_PROMPTS[best_idx]
    best_prob = float(probs[best_idx])
    return asian_score, best_prompt, best_prob

def clip_child_adult_elderly(img_bgr):
    probs = _clip_probs(img_bgr, AGESTYLE_PROMPTS)
    child_score   = float(probs[0] + probs[1] + probs[2] + probs[3])
    adult_score   = float(probs[4])
    elderly_score = float(probs[5])
    return child_score, adult_score, elderly_score

# ================== Age bin helper ==================
def get_age_bin(age: float):
    if age is None or age < 0:
        return None
    if age <= 8: return "CHILD8"
    if 9 <= age <= 18: return "9-18"
    if 19 <= age <= 45: return "19-45"
    if 46 <= age <= 65: return "46-65"
    if age >= 66: return "Elderly"
    return None

# ================== MAIN ==================
scanned = 0
asian_pass = 0
no_face = 0
unknown_age = 0
unknown_gender = 0

kept_counts = {f"{b}/{g}": 0 for (b,_,_) in AGE_BINS for g in ["Male","Female"]}
rows = []

reject_child8 = 0
reject_adult_childlike = 0
reject_adult_notadult = 0
reject_elderly = 0

for path in sorted(glob.glob(IN_GLOB, recursive=True)):
    scanned += 1
    img = cv2.imread(path)
    if img is None:
        continue

    # Asian gate
    asian_score, best_prompt, best_prob = clip_asian_score(img)
    if asian_score < ASIAN_CONF_MIN:
        continue
    asian_pass += 1

    # Age/Gender
    ag = infer_age_gender(img)
    if ag is None:
        no_face += 1
        continue
    age, gender = ag

    # gender required
    if gender == "M":
        gender_folder = "Male"
    elif gender == "F":
        gender_folder = "Female"
    else:
        unknown_gender += 1
        continue

    bin_name = get_age_bin(age)
    if bin_name is None:
        unknown_age += 1
        continue

    # CLIP age-style
    child_score, adult_score, elderly_score = clip_child_adult_elderly(img)
    note = f"child={child_score:.3f} adult={adult_score:.3f} elderly={elderly_score:.3f}"

    # Tight CHILD8
    if bin_name == "CHILD8":
        if not (child_score >= CHILDLIKE_MIN and adult_score <= ADULTLIKE_MAX_IN_CHILD):
            reject_child8 += 1
            rows.append([os.path.basename(path), parse_seed(path), age, gender, asian_score, ASIAN_CONF_MIN,
                         best_prompt, best_prob, bin_name, None, note, "REJECT_CHILD8_TIGHT"])
            continue

    # Anti-child gate for adult bins: chặn 9-18 lọt vào 19-45 / 46-65 / Elderly
    if bin_name in ADULT_BINS:
        if child_score > ADULT_CHILD_MAX:
            reject_adult_childlike += 1
            rows.append([os.path.basename(path), parse_seed(path), age, gender, asian_score, ASIAN_CONF_MIN,
                         best_prompt, best_prob, bin_name, None, note, "REJECT_ADULT_CHILDLIKE"])
            continue
        if adult_score < ADULTLIKE_MIN:
            reject_adult_notadult += 1
            rows.append([os.path.basename(path), parse_seed(path), age, gender, asian_score, ASIAN_CONF_MIN,
                         best_prompt, best_prob, bin_name, None, note, "REJECT_ADULT_NOTADULT"])
            continue

    # Elderly tighten (nhẹ)
    if bin_name == "Elderly":
        if elderly_score < ELDERLYLIKE_MIN and adult_score > 0.50:
            reject_elderly += 1
            rows.append([os.path.basename(path), parse_seed(path), age, gender, asian_score, ASIAN_CONF_MIN,
                         best_prompt, best_prob, bin_name, None, note, "REJECT_ELDERLY"])
            continue

    # Copy
    base = os.path.basename(path)
    dst = os.path.join(OUTROOT, bin_name, gender_folder, base)
    shutil.copy2(path, dst)
    kept_counts[f"{bin_name}/{gender_folder}"] += 1
    rows.append([base, parse_seed(path), age, gender, asian_score, ASIAN_CONF_MIN,
                 best_prompt, best_prob, bin_name, dst, note, "OK"])

print("Scanned:", scanned)
print("Asian passed:", asian_pass, "min=", ASIAN_CONF_MIN)
print("No face detected:", no_face)
print("Unknown age:", unknown_age, "| Unknown gender:", unknown_gender)
print("Reject CHILD8 tight:", reject_child8)
print("Reject adult (childlike):", reject_adult_childlike)
print("Reject adult (not adult):", reject_adult_notadult)
print("Reject elderly:", reject_elderly)
print("Output root:", OUTROOT)

for k, v in kept_counts.items():
    if v > 0:
        print(f"{k}: {v}")

with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow([
        "file","seed","age_est","gender",
        "asian_score","asian_conf_min",
        "best_prompt","best_prob",
        "age_bin","dst_path",
        "clip_note","status"
    ])
    w.writerows(rows)

print("Wrote:", CSV_PATH)


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition
set det-size: (640, 640)
Scanned: 501
Asian passed: 218 min= 0.05
No face detected: 145
Unknown age: 0 | Unknown gender: 0
Re

In [ ]:
#Lấy cả, không lọc Asian

In [66]:
!rm -rf /content/filtered_all

In [47]:
!pip -q install -U mediapipe onnxruntime-gpu transformers opencv-python


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 16.0 MB/s eta 0:00:00


In [48]:
import os, urllib.request

MP_MODEL_PATH = "/content/blaze_face_short_range.tflite"
if not os.path.exists(MP_MODEL_PATH):
    url = "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/latest/blaze_face_short_range.tflite"
    urllib.request.urlretrieve(url, MP_MODEL_PATH)

print("MP model:", MP_MODEL_PATH, "exists:", os.path.exists(MP_MODEL_PATH))


MP model: /content/blaze_face_short_range.tflite exists: True


In [50]:
import os, urllib.request

GA_PATH = "/content/genderage.onnx"

URLS = [
    # GitHub release (có thể fail)
    "https://github.com/deepinsight/insightface/releases/download/v0.7/genderage.onnx",
    # Mirror trên HuggingFace (thường ổn định hơn)
    "https://huggingface.co/deepinsight/insightface/resolve/main/models/buffalo_l/genderage.onnx",
    "https://huggingface.co/deepinsight/insightface/resolve/main/genderage.onnx",
]

if not os.path.exists(GA_PATH):
    ok = False
    for url in URLS:
        try:
            print("Trying:", url)
            urllib.request.urlretrieve(url, GA_PATH)
            if os.path.getsize(GA_PATH) > 1_000_000:  # sanity: >1MB
                ok = True
                print("Downloaded:", GA_PATH, "size:", os.path.getsize(GA_PATH))
                break
        except Exception as e:
            print("  failed:", repr(e))

    if not ok:
        print("\n❌ Không tải được genderage.onnx từ các nguồn tự động.")
        print("➡️ Cách 1: Upload file genderage.onnx lên /content rồi đặt GA_PATH đúng.")
        print("➡️ Cách 2: Upload lên Google Drive rồi trỏ GA_PATH tới file đó.")
else:
    print("Found existing:", GA_PATH, "size:", os.path.getsize(GA_PATH))


Found existing: /content/genderage.onnx size: 1322532


In [67]:
# ==========================================================
# FULL PIPELINE (Age bins OK + Gender robust)
# - MediaPipe Tasks: detect face -> crop
# - InsightFace: age (vẫn dùng logic tuổi như trước)
# - Gender robust: InsightFace x2 crop (scale 1.05 & 1.25) + CLIP fallback
# - Bins: CHILD8, 9-18, 19-45, 46-65, Elderly
# - Output: OUTROOT/<BIN>/<Male|Female>/...
# - CSV: selected_all.csv (kèm debug scores)
# ==========================================================

# !pip -q install -U mediapipe insightface onnxruntime transformers opencv-python

import os, re, csv, shutil, glob
import cv2
import numpy as np

# ================== INPUT ==================
IN_GLOB = "/content/100_test/**/*.png"

# ================== OUTPUT ==================
OUTROOT  = "/content/filtered_all"
CSV_PATH = os.path.join(OUTROOT, "selected_all.csv")
CLEAR_OUTROOT_BEFORE_RUN = False  # True nếu muốn xoá output cũ mỗi lần chạy

# ================== BINS ==================
AGE_BINS = [
    ("CHILD8",   0,   8),
    ("9-18",     9,  18),
    ("19-45",   19,  45),
    ("46-65",   46,  65),
    ("Elderly", 66, 200),
]

def get_age_bin(age: float):
    if age is None or age < 0:
        return None
    if age <= 8: return "CHILD8"
    if 9 <= age <= 18: return "9-18"
    if 19 <= age <= 45: return "19-45"
    if 46 <= age <= 65: return "46-65"
    if age >= 66: return "Elderly"
    return None

# ================== UTIL ==================
seed_re = re.compile(r"(\d+)")
def parse_seed(path):
    base = os.path.basename(path)
    m = seed_re.findall(base)
    return int(m[0]) if m else None

def ensure_dirs():
    os.makedirs(OUTROOT, exist_ok=True)
    for bin_name, _, _ in AGE_BINS:
        os.makedirs(os.path.join(OUTROOT, bin_name, "Male"), exist_ok=True)
        os.makedirs(os.path.join(OUTROOT, bin_name, "Female"), exist_ok=True)

def clear_outroot():
    if os.path.exists(OUTROOT):
        shutil.rmtree(OUTROOT, ignore_errors=True)

if CLEAR_OUTROOT_BEFORE_RUN:
    clear_outroot()
ensure_dirs()

# ================== MediaPipe Tasks FaceDetector ==================
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import urllib.request

MP_SCORE_MIN   = 0.50
MP_SCALE_AGE   = 1.15     # crop "mặt" hơn để age ổn định
MIN_CROP_SIZE  = 140

MP_MODEL_PATH = "/content/blaze_face_short_range.tflite"
if not os.path.exists(MP_MODEL_PATH):
    url = "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/latest/blaze_face_short_range.tflite"
    urllib.request.urlretrieve(url, MP_MODEL_PATH)

base_options = python.BaseOptions(model_asset_path=MP_MODEL_PATH)
options = vision.FaceDetectorOptions(base_options=base_options, min_detection_confidence=MP_SCORE_MIN)
mp_detector = vision.FaceDetector.create_from_options(options)

def mp_detect_crop(img_bgr, scale=1.15):
    h, w = img_bgr.shape[:2]
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    res = mp_detector.detect(mp_image)
    dets = res.detections if res and res.detections else []
    if not dets:
        return None

    # pick largest face
    best = None
    best_area = -1
    for det in dets:
        bb = det.bounding_box
        x1, y1, bw, bh = int(bb.origin_x), int(bb.origin_y), int(bb.width), int(bb.height)
        area = max(0, bw) * max(0, bh)
        if area > best_area:
            best_area = area
            best = (x1, y1, bw, bh)
    if best is None:
        return None

    x1, y1, bw, bh = best
    cx, cy = x1 + bw/2.0, y1 + bh/2.0
    bw2, bh2 = bw*scale, bh*scale

    nx1 = int(max(0, cx - bw2/2.0))
    ny1 = int(max(0, cy - bh2/2.0))
    nx2 = int(min(w, cx + bw2/2.0))
    ny2 = int(min(h, cy + bh2/2.0))

    if nx2 <= nx1 or ny2 <= ny1:
        return None
    crop = img_bgr[ny1:ny2, nx1:nx2]
    if crop.size == 0:
        return None
    if min(crop.shape[:2]) < MIN_CROP_SIZE:
        return None
    return crop

# ================== InsightFace (age + gender) ==================
from insightface.app import FaceAnalysis

ifa = FaceAnalysis(name="buffalo_l", allowed_modules=["detection","genderage"])
ifa.prepare(ctx_id=0, det_size=(320,320), det_thresh=0.3)

def infer_age_gender_from_crop(crop_bgr):
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    faces = ifa.get(crop_rgb)
    if not faces:
        return None
    face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
    age = float(getattr(face, "age", -1))
    gender_id = int(getattr(face, "gender", -1))  # 0 female, 1 male
    gender = "F" if gender_id == 0 else ("M" if gender_id == 1 else "U")
    return age, gender

def infer_gender_if_only(crop_bgr):
    ag = infer_age_gender_from_crop(crop_bgr)
    if ag is None:
        return None
    _, g = ag
    return g if g in ("M","F") else None

# ================== CLIP fallback for gender ==================
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

GENDER_PROMPTS = [
    "a photo of a male person",
    "a photo of a female person",
]

def clip_gender_prob(img_bgr):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    inputs = proc(text=GENDER_PROMPTS, images=pil, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        probs = clip(**inputs).logits_per_image[0].softmax(dim=-1).detach().cpu().numpy()
    pm, pf = float(probs[0]), float(probs[1])
    return pm, pf

def infer_gender_robust(img_bgr_full):
    """
    Gender robust:
    - crop scale 1.05 and 1.25
    - if both IF agree => accept
    - else CLIP confirm if confident
    - else fallback to available IF or 'U'
    """
    crop_a = mp_detect_crop(img_bgr_full, scale=1.05)
    crop_b = mp_detect_crop(img_bgr_full, scale=1.25)

    g1 = infer_gender_if_only(crop_a) if crop_a is not None else None
    g2 = infer_gender_if_only(crop_b) if crop_b is not None else None

    if g1 is not None and g1 == g2:
        return g1, f"IFx2={g1}"

    crop_for_clip = crop_a if crop_a is not None else (crop_b if crop_b is not None else None)
    if crop_for_clip is None:
        return "U", "NO_CROP_FOR_GENDER"

    pm, pf = clip_gender_prob(crop_for_clip)
    clip_g = "M" if pm > pf else "F"
    clip_conf = max(pm, pf)

    # if CLIP strongly agrees with any IF vote -> accept that
    if g1 is not None and clip_g == g1 and clip_conf >= 0.60:
        return g1, f"IF={g1},CLIP={clip_g}({clip_conf:.2f})"
    if g2 is not None and clip_g == g2 and clip_conf >= 0.60:
        return g2, f"IF={g2},CLIP={clip_g}({clip_conf:.2f})"

    # If CLIP very confident but IF disagrees, safer set Unknown to avoid wrong labels
    if clip_conf >= 0.70 and (g1 is not None or g2 is not None):
        return "U", f"CONFLICT IF({g1},{g2}) vs CLIP={clip_g}({clip_conf:.2f})"

    # fallback IF if any
    g = g1 if g1 is not None else (g2 if g2 is not None else "U")
    return g, f"WEAK IF({g1},{g2}) CLIP={clip_g}({clip_conf:.2f})"

# ================== MAIN ==================
scanned = 0
no_crop = 0
ifa_no_face = 0
bad_age = 0
unknown_gender = 0

kept_counts = {f"{b}/{g}": 0 for (b,_,_) in AGE_BINS for g in ["Male","Female"]}
rows = []

for path in sorted(glob.glob(IN_GLOB, recursive=True)):
    scanned += 1
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        continue

    crop = mp_detect_crop(img, scale=MP_SCALE_AGE)
    if crop is None:
        no_crop += 1
        rows.append([os.path.basename(path), parse_seed(path), None, None, None, None, "NO_CROP"])
        continue

    ag = infer_age_gender_from_crop(crop)
    if ag is None:
        ifa_no_face += 1
        rows.append([os.path.basename(path), parse_seed(path), None, None, None, None, "IFA_NO_FACE_ON_CROP"])
        continue

    age, gender_if = ag
    if not (0 <= age <= 120):
        bad_age += 1
        rows.append([os.path.basename(path), parse_seed(path), age, gender_if, None, None, "BAD_AGE"])
        continue

    bin_name = get_age_bin(age)
    if bin_name is None:
        bad_age += 1
        rows.append([os.path.basename(path), parse_seed(path), age, gender_if, None, None, "UNKNOWN_AGEBIN"])
        continue

    # robust gender (uses full image for multi-crop)
    gender, gender_meta = infer_gender_robust(img)
    if gender not in ("M","F"):
        unknown_gender += 1
        rows.append([os.path.basename(path), parse_seed(path), age, gender, bin_name, None, "UNKNOWN_GENDER | "+gender_meta])
        continue

    gender_folder = "Male" if gender == "M" else "Female"
    base = os.path.basename(path)
    dst = os.path.join(OUTROOT, bin_name, gender_folder, base)
    shutil.copy2(path, dst)
    kept_counts[f"{bin_name}/{gender_folder}"] += 1

    rows.append([base, parse_seed(path), age, gender, bin_name, dst, f"OK | IF_gender={gender_if} | {gender_meta}"])

print("Scanned:", scanned)
print("No crop:", no_crop)
print("InsightFace no face on crop:", ifa_no_face)
print("Bad age:", bad_age)
print("Unknown gender:", unknown_gender)
print("Output root:", OUTROOT)
for k,v in kept_counts.items():
    if v>0:
        print(f"{k}: {v}")

with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["file","seed","age_est","gender_final","age_bin","dst_path","status_note"])
    w.writerows(rows)

print("Wrote:", CSV_PATH)


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition
set det-size: (320, 320)
Scanned: 501
No crop: 0
InsightFace no face on crop: 0
Bad age: 0
Unknown gender: 0
Output root: /co

In [61]:
import pandas as pd
df = pd.read_csv("/content/filtered_all/selected_all.csv")
print(df["age_est"].describe())
print("<=8:", (df["age_est"]<=8).sum())
print("<=10:", (df["age_est"]<=10).sum())
print("<=12:", (df["age_est"]<=12).sum())


count    501.000000
mean      38.632735
std        9.205262
min       17.000000
25%       31.000000
50%       37.000000
75%       44.000000
max       68.000000
Name: age_est, dtype: float64
<=8: 0
<=10: 0
<=12: 0
